In [37]:
# Cell 1: 라이브러리 가져오기
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import glob # 여러 파일을 다루기 위한 라이브러리

print(f"PyTorch Version: {torch.__version__}")

PyTorch Version: 2.6.0+cu124


In [38]:
# Cell 2: GPU 장치 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
print(f"CUDA 버전: {torch.version.cuda}")
print(f"GPU 개수: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU 이름: {torch.cuda.get_device_name(0)}")

Using device: cuda
CUDA 사용 가능: True
CUDA 버전: 12.4
GPU 개수: 1
GPU 이름: NVIDIA GeForce RTX 4080 SUPER


In [39]:
# Cell 3: 데이터 로딩 및 전처리
# 문자열 레이블을 숫자로 바꾸기 위한 LabelEncoder를 가져옵니다.
from sklearn.preprocessing import StandardScaler, LabelEncoder

# .ipynb 파일 위치를 기준으로 'csv' 폴더 안의 모든 .csv 파일을 찾아옵니다.
file_paths = glob.glob('csv/*.csv')

# 만약 파일이 하나도 찾아지지 않을 경우를 대비한 확인 코드
if not file_paths:
    print("오류: 'csv' 폴더에서 CSV 파일을 찾을 수 없습니다. 폴더 이름과 파일 위치를 다시 확인해주세요.")
else:
    print(f"총 {len(file_paths)}개의 CSV 파일을 찾았습니다.")
    print(file_paths[0]) # 첫 번째로 찾은 파일 경로 출력

    # 여러 CSV 파일을 하나의 데이터프레임으로 읽어오기
    df_list = [pd.read_csv(file) for file in file_paths]
    df = pd.concat(df_list, ignore_index=True)

    print("\n--- 데이터 샘플 (상위 5개) ---")
    print(df.head())
    print("\n--- 데이터 정보 ---")
    df.info()

    # --- 데이터 전처리 ---
    X = df.iloc[:, :-1].values
    y_str = df.iloc[:, -1].values # 원본 '문자열' 레이블을 y_str 이라는 새 변수에 저장

    # LabelEncoder를 사용하여 문자열 레이블을 숫자로 변환
    encoder = LabelEncoder()
    y = encoder.fit_transform(y_str) # 최종 y는 이제 숫자 배열

    # 어떤 문자가 어떤 숫자로 변환되었는지 직접 눈으로 확인할 수 있도록 출력
    print("\n--- 레이블 인코딩 결과 ---")
    print(f"원본 레이블 종류: {encoder.classes_}")
    print(f"숫자로 변환된 레이블 종류: {np.unique(y)}")
    print(f"예시: 원본 '{y_str[0]}' 가 숫자 '{y[0]}' (으)로 변환되었습니다.")

    # 특성 데이터 정규화
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # y 데이터의 형태를 확인하는 print문을 추가하여 최종 결과를 명확히 함
    print(f"\n원본 특성 데이터 형태: {X.shape}")
    print(f"정규화된 특성 데이터 형태: {X_scaled.shape}")
    print(f"인코딩된 레이블 데이터 형태: {y.shape}")

총 20개의 CSV 파일을 찾았습니다.
csv\modified_perturbed_synthetic_wafer_dataset.csv

--- 데이터 샘플 (상위 5개) ---
     Sensor-1    Sensor-2    Sensor-3    Sensor-4    Sensor-5    Sensor-6  \
0  102.026080  101.554858  103.177063  101.671363  101.982248  103.313323   
1  103.132429  103.358821  101.908280  102.340709  102.660106  103.087193   
2  102.650988  102.925712  101.686257  103.287270  102.941505  103.398055   
3  102.356317  103.643141  103.286108  102.457860  103.611915  103.064590   
4  102.066037  104.144133  103.563858  103.499183  102.815099  103.234462   

     Sensor-7    Sensor-8    Sensor-9   Sensor-10  ...  Sensor-434  \
0  102.031870  102.785179  102.812477  104.367773  ...  111.911140   
1  102.079181  102.238647  101.155078  103.394064  ...  111.848585   
2  101.480021  101.620209  102.520577  103.298871  ...  112.470402   
3  101.937405  102.879829  103.300430  104.345083  ...  112.237401   
4  101.909272  101.846750  102.560261  104.173849  ...  113.398781   

   Sensor-435  Sens

In [40]:
# Cell 4: PyTorch Custom Dataset 정의
class CSVDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long) # 분류 문제일 경우 long 타입 사용

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        # 1D CNN 입력을 위해 채널 차원 추가 (batch, channels, sequence_length)
        # 현재는 채널이 1개, sequence_length는 특성의 개수
        feature = self.features[idx].unsqueeze(0)
        label = self.labels[idx]
        return feature, label

# 데이터셋 생성
dataset = CSVDataset(X_scaled, y)
print(f"\n첫 번째 데이터 샘플 형태: {dataset[0][0].shape}")


첫 번째 데이터 샘플 형태: torch.Size([1, 442])


In [41]:
# Cell 5: DataLoader 생성 (훈련/검증/테스트 데이터 분리)
if 'df' in locals():
    dataset = CSVDataset(X_scaled, y)
    print(f"전체 데이터셋 크기: {len(dataset)}")

    # ### 수정된 부분 ### : 데이터를 훈련, 검증, 테스트 세트로 나눌 비율 정의
    TRAIN_RATIO = 0.7
    VALID_RATIO = 0.15
    TEST_RATIO = 0.15

    # 데이터셋 분리
    n_total = len(dataset)
    n_train = int(n_total * TRAIN_RATIO)
    n_valid = int(n_total * VALID_RATIO)
    n_test = n_total - n_train - n_valid # 나머지 모두 테스트용

    train_dataset, valid_dataset, test_dataset = random_split(dataset, [n_train, n_valid, n_test])

    # DataLoader 생성
    BATCH_SIZE = 64
    train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    valid_loader = DataLoader(dataset=valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False) # 테스트 로더 추가

    print(f"\n학습 데이터: {len(train_dataset)}개, 검증 데이터: {len(valid_dataset)}개, 테스트 데이터: {len(test_dataset)}개")

전체 데이터셋 크기: 2000

학습 데이터: 1400개, 검증 데이터: 300개, 테스트 데이터: 300개


In [42]:
# Cell 6: 1D CNN 모델 정의
class Simple1DCNN(nn.Module):
    def __init__(self, num_features, num_classes):
        super(Simple1DCNN, self).__init__()
        self.layer1 = nn.Sequential(
            nn.Conv1d(in_channels=1, out_channels=16, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2)
        )
        self.layer2 = nn.Sequential(
            nn.Conv1d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2)
        )
        
        # MaxPool1d를 거친 후의 데이터 크기를 계산해야 합니다.
        # 초기 특성 수: num_features
        # layer1 후: floor(num_features / 2)
        # layer2 후: floor(floor(num_features / 2) / 2)
        flattened_size = 32 * (num_features // 4)

        # Fully Connected Layer에 Dropout을 추가합니다.
        self.fc = nn.Sequential(
            nn.Linear(flattened_size, 128), # 중간 레이어를 하나 추가해볼 수 있습니다.
            nn.ReLU(),
            nn.Dropout(0.5), # 50%의 뉴런을 랜덤으로 비활성화합니다.
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = x.view(x.size(0), -1)
        out = self.fc(x)
        return out

# 모델 인스턴스 생성 및 GPU로 이동
if 'df' in locals():
    num_features = X.shape[1]
    num_classes = len(np.unique(y))
    model = Simple1DCNN(num_features=num_features, num_classes=num_classes).to(device)

    print("--- 수정된 모델 구조 (Dropout 포함) ---")
    print(model)

--- 수정된 모델 구조 (Dropout 포함) ---
Simple1DCNN(
  (layer1): Sequential(
    (0): Conv1d(1, 16, kernel_size=(3,), stride=(1,), padding=(1,))
    (1): ReLU()
    (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (layer2): Sequential(
    (0): Conv1d(16, 32, kernel_size=(3,), stride=(1,), padding=(1,))
    (1): ReLU()
    (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc): Sequential(
    (0): Linear(in_features=3520, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.5, inplace=False)
    (3): Linear(in_features=128, out_features=8, bias=True)
  )
)


In [43]:
# Cell 7: 손실 함수 및 옵티마이저 정의
# 하이퍼파라미터
LEARNING_RATE = 0.00005

criterion = nn.CrossEntropyLoss() # 다중 분류 문제이므로 CrossEntropyLoss 사용
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [44]:
# Cell 8: 모델 학습 루프
# 하이퍼파라미터
NUM_EPOCHS = 20

for epoch in range(NUM_EPOCHS):
    model.train() # 학습 모드
    train_loss = 0.0
    for i, (features, labels) in enumerate(train_loader):
        # 데이터와 레이블을 GPU로 이동
        features = features.to(device)
        labels = labels.to(device)

        # 순전파
        outputs = model(features)
        loss = criterion(outputs, labels)

        # 역전파 및 가중치 업데이트
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # 검증
    model.eval() # 평가 모드
    valid_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad(): # 기울기 계산 비활성화
        for features, labels in valid_loader:
            features = features.to(device)
            labels = labels.to(device)

            outputs = model(features)
            loss = criterion(outputs, labels)
            valid_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}], "
          f"Train Loss: {train_loss/len(train_loader):.4f}, "
          f"Valid Loss: {valid_loss/len(valid_loader):.4f}, "
          f"Accuracy: {accuracy:.2f}%")

print("\n학습 완료!")

Epoch [1/20], Train Loss: 2.0117, Valid Loss: 1.9318, Accuracy: 32.00%
Epoch [2/20], Train Loss: 1.8575, Valid Loss: 1.7725, Accuracy: 42.00%
Epoch [3/20], Train Loss: 1.6803, Valid Loss: 1.5965, Accuracy: 66.33%
Epoch [4/20], Train Loss: 1.4910, Valid Loss: 1.4183, Accuracy: 75.67%
Epoch [5/20], Train Loss: 1.3145, Valid Loss: 1.2393, Accuracy: 88.33%
Epoch [6/20], Train Loss: 1.1531, Valid Loss: 1.0709, Accuracy: 91.33%
Epoch [7/20], Train Loss: 0.9825, Valid Loss: 0.9155, Accuracy: 92.33%
Epoch [8/20], Train Loss: 0.8343, Valid Loss: 0.7735, Accuracy: 95.33%
Epoch [9/20], Train Loss: 0.7230, Valid Loss: 0.6508, Accuracy: 97.33%
Epoch [10/20], Train Loss: 0.6153, Valid Loss: 0.5472, Accuracy: 98.33%
Epoch [11/20], Train Loss: 0.5210, Valid Loss: 0.4581, Accuracy: 100.00%
Epoch [12/20], Train Loss: 0.4380, Valid Loss: 0.3842, Accuracy: 100.00%
Epoch [13/20], Train Loss: 0.3761, Valid Loss: 0.3221, Accuracy: 100.00%
Epoch [14/20], Train Loss: 0.3300, Valid Loss: 0.2740, Accuracy: 100.0

In [45]:
# Cell 9: 최종 모델 성능 평가 (Test)
if 'df' in locals():
    model.eval() # 모델을 평가 모드로 설정
    test_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad(): # 기울기 계산 비활성화
        for features, labels in test_loader: # ### test_loader 사용 ###
            features = features.to(device)
            labels = labels.to(device)

            outputs = model(features)
            loss = criterion(outputs, labels)
            test_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print("------------------------")
    print("      최종 모델 성능     ")
    print("------------------------")
    print(f"Test Loss: {test_loss/len(test_loader):.4f}")
    print(f"Test Accuracy: {accuracy:.2f}%")

------------------------
      최종 모델 성능     
------------------------
Test Loss: 0.0976
Test Accuracy: 100.00%
